In [5]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
        .appName ('TidarMart')     
        .master('local[2]')  
        .config('spark.driver.memory', '1536m')     
        .config('spark.sql.shuffle.partitions', '4')     
        .config('spark.sql.adaptive.enabled', 'true')    
        .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

SparkSession berhasil dibuat!
Versi Spark: 3.5.9


**Tahap 1 – Penyimpanan Terdistribusi dan Ingesti (HDFS)**

In [6]:
!hdfs dfs -mkdir -p /tidarmart/raw/transaksi
!hdfs dfs -mkdir -p /tidarmart/raw/pelanggan
!hdfs dfs -mkdir -p /tidarmart/raw/produk
!hdfs dfs -mkdir -p /tidarmart/raw/strem_input
!hdfs dfs -mkdir -p /tidarmart/raw/curated
!hdfs dfs -mkdir -p /tidarmart/raw/models

In [7]:
!hdfs dfs -ls -R /tidarmart

drwxr-xr-x   - fahma supergroup          0 2026-11-26 08:50 /tidarmart/raw
drwxr-xr-x   - fahma supergroup          0 2026-11-26 07:59 /tidarmart/raw/curated
drwxr-xr-x   - fahma supergroup          0 2026-11-26 08:00 /tidarmart/raw/models
drwxr-xr-x   - fahma supergroup          0 2026-11-26 08:08 /tidarmart/raw/pelanggan
-rw-r--r--   1 fahma supergroup    6307840 2026-11-26 08:08 /tidarmart/raw/pelanggan/pelanggan.json
drwxr-xr-x   - fahma supergroup          0 2026-11-26 08:08 /tidarmart/raw/produk
-rw-r--r--   1 fahma supergroup     700615 2026-11-26 08:08 /tidarmart/raw/produk/produk.json
drwxr-xr-x   - fahma supergroup          0 2026-11-26 07:59 /tidarmart/raw/stream_input
drwxr-xr-x   - fahma supergroup          0 2026-11-26 08:50 /tidarmart/raw/strem_input
drwxr-xr-x   - fahma supergroup          0 2026-11-26 08:07 /tidarmart/raw/transaksi
-rw-r--r--   1 fahma supergroup   49068656 2026-11-26 08:07 /tidarmart/raw/transaksi/transaksi_part1.csv
-rw-r--r--   1 fahma supergroup   

In [20]:
!hdfs dfs -put -f /home/fahma/praktikum-bigdata/proyek-akhir/data_proyek/transaksi/*.csv /tidarmart/raw/transaksi/
!hdfs dfs -put -f /home/fahma/praktikum-bigdata/proyek-akhir/data_proyek/pelanggan.json /tidarmart/raw/pelanggan/
!hdfs dfs -put -f /home/fahma/praktikum-bigdata/proyek-akhir/data_proyek/produk.json /tidarmart/raw/produk/
!hdfs dfs -put -f /home/fahma/praktikum-bigdata/proyek-akhir/data_proyek/stream_input/*.csv /tidarmart/raw/stream_input/

In [18]:
pwd

'/home/fahma/praktikum-bigdata/proyek-akhir'

In [21]:
!hdfs dfs -ls -R /tidarmart

drwxr-xr-x   - fahma supergroup          0 2026-11-26 08:50 /tidarmart/raw
drwxr-xr-x   - fahma supergroup          0 2026-11-26 07:59 /tidarmart/raw/curated
drwxr-xr-x   - fahma supergroup          0 2026-11-26 08:00 /tidarmart/raw/models
drwxr-xr-x   - fahma supergroup          0 2026-11-26 09:10 /tidarmart/raw/pelanggan
-rw-r--r--   1 fahma supergroup    6307840 2026-11-26 09:10 /tidarmart/raw/pelanggan/pelanggan.json
drwxr-xr-x   - fahma supergroup          0 2026-11-26 09:10 /tidarmart/raw/produk
-rw-r--r--   1 fahma supergroup     700615 2026-11-26 09:10 /tidarmart/raw/produk/produk.json
drwxr-xr-x   - fahma supergroup          0 2026-11-26 09:10 /tidarmart/raw/stream_input
-rw-r--r--   1 fahma supergroup     490170 2026-11-26 09:10 /tidarmart/raw/stream_input/stream_01.csv
-rw-r--r--   1 fahma supergroup     490310 2026-11-26 09:10 /tidarmart/raw/stream_input/stream_02.csv
-rw-r--r--   1 fahma supergroup     490723 2026-11-26 09:10 /tidarmart/raw/stream_input/stream_03.csv
-rw-r

In [22]:
!hdfs dfs -ls /tidarmart/raw/transaksi
!hdfs dfs -ls /tidarmart/raw/pelanggan
!hdfs dfs -ls /tidarmart/raw/produk
!hdfs dfs -ls /tidarmart/raw/stream_input

Found 4 items
-rw-r--r--   1 fahma supergroup   49068656 2026-11-26 09:10 /tidarmart/raw/transaksi/transaksi_part1.csv
-rw-r--r--   1 fahma supergroup   49063997 2026-11-26 09:10 /tidarmart/raw/transaksi/transaksi_part2.csv
-rw-r--r--   1 fahma supergroup   49062882 2026-11-26 09:10 /tidarmart/raw/transaksi/transaksi_part3.csv
-rw-r--r--   1 fahma supergroup   49070313 2026-11-26 09:10 /tidarmart/raw/transaksi/transaksi_part4.csv
Found 1 items
-rw-r--r--   1 fahma supergroup    6307840 2026-11-26 09:10 /tidarmart/raw/pelanggan/pelanggan.json
Found 1 items
-rw-r--r--   1 fahma supergroup     700615 2026-11-26 09:10 /tidarmart/raw/produk/produk.json
Found 20 items
-rw-r--r--   1 fahma supergroup     490170 2026-11-26 09:10 /tidarmart/raw/stream_input/stream_01.csv
-rw-r--r--   1 fahma supergroup     490310 2026-11-26 09:10 /tidarmart/raw/stream_input/stream_02.csv
-rw-r--r--   1 fahma supergroup     490723 2026-11-26 09:10 /tidarmart/raw/stream_input/stream_03.csv
-rw-r--r--   1 fahma su

In [23]:
!hdfs dfs -du -h/tidarmart

-du: Illegal option -h/tidarmart
Usage: hadoop fs [generic options]
	[-appendToFile [-n] <localsrc> ... <dst>]
	[-cat [-ignoreCrc] <src> ...]
	[-checksum [-v] <src> ...]
	[-chgrp [-R] GROUP PATH...]
	[-chmod [-R] <MODE[,MODE]... | OCTALMODE> PATH...]
	[-chown [-R] [OWNER][:[GROUP]] PATH...]
	[-concat <target path> <src path> <src path> ...]
	[-copyFromLocal [-f] [-p] [-l] [-d] [-t <thread count>] [-q <thread pool queue size>] <localsrc> ... <dst>]
	[-copyToLocal [-f] [-p] [-crc] [-ignoreCrc] [-t <thread count>] [-q <thread pool queue size>] <src> ... <localdst>]
	[-count [-q] [-h] [-v] [-t [<storage type>]] [-u] [-x] [-e] [-s] <path> ...]
	[-cp [-f] [-p | -p[topax]] [-d] [-t <thread count>] [-q <thread pool queue size>] <src> ... <dst>]
	[-createSnapshot <snapshotDir> [<snapshotName>]]
	[-deleteSnapshot <snapshotDir> <snapshotName>]
	[-df [-h] [<path> ...]]
	[-du [-s] [-h] [-v] [-x] <path> ...]
	[-expunge [-immediate] [-fs <path>]]
	[-find <path> ... <expression> ...]
	[-get [-f] [-p] 

In [26]:
!hdfs fsck /tidarmart -files -blocks -locations

Connecting to namenode via http://localhost:9870/fsck?ugi=fahma&files=1&blocks=1&locations=1&path=%2Ftidarmart
FSCK started by fahma (auth:SIMPLE) from /127.0.0.1 for path /tidarmart at Thu Nov 26 09:16:47 WIB 2026

/tidarmart <dir>
/tidarmart/raw <dir>
/tidarmart/raw/curated <dir>
/tidarmart/raw/models <dir>
/tidarmart/raw/pelanggan <dir>
/tidarmart/raw/pelanggan/pelanggan.json 6307840 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-111730338-127.0.1.1-1795610580917:blk_1073741850_1026 len=6307840 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-6e060d9d-c7c0-453f-858e-f2e75af20d5d,DISK]]

/tidarmart/raw/produk <dir>
/tidarmart/raw/produk/produk.json 700615 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-111730338-127.0.1.1-1795610580917:blk_1073741851_1027 len=700615 Live_repl=1  [DatanodeInfoWithStorage[127.0.0.1:9866,DS-6e060d9d-c7c0-453f-858e-f2e75af20d5d,DISK]]

/tidarmart/raw/stream_input <dir>
/tidarmart/raw/stream_input/stream_01.csv 490170 bytes, replic

In [38]:
HDFS = "hdfs://localhost:9000/tidarmart"
df_trx = spark.read.csv(f"{HDFS}/raw/transaksi/*.csv", header=True)
print("Jumlah baris:", df_trx.count())
df_trx.printSchema()
df_trx.show(5)

Jumlah baris: 2010000
root
 |-- order_id: string (nullable = true)
 |-- tanggal: string (nullable = true)
 |-- pelanggan_id: string (nullable = true)
 |-- produk_id: string (nullable = true)
 |-- qty: string (nullable = true)
 |-- harga_satuan: string (nullable = true)
 |-- diskon_persen: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- berat_kg: string (nullable = true)
 |-- jarak_km: string (nullable = true)
 |-- lama_kirim_hari: string (nullable = true)
 |-- status_pesanan: string (nullable = true)

+----------+-------------------+------------+---------+---+------------+-------------+-----------------+----------+--------+--------+---------------+--------------+
|  order_id|            tanggal|pelanggan_id|produk_id|qty|harga_satuan|diskon_persen|metode_pembayaran|      kota|berat_kg|jarak_km|lama_kirim_hari|status_pesanan|
+----------+-------------------+------------+---------+---+------------+-------------+----------

In [43]:
from pyspark.sql.types import StructType, StructField, StringType
skema_pel = StructType([
    StructField("pelanggan_id", StringType()),
    StructField("nama", StringType()),
    StructField("kota", StringType()),
    StructField("tanggal_daftar", StringType()),
    StructField("membership", StringType()),
])

In [49]:
df_pel = spark.read.schema(skema_pel).json(f"{HDFS}/raw/pelanggan/pelanggan.json")
print("Jumlah pelanggan:", df_pel.count())
df_pel.printSchema()
df_pel.show(7, truncate=False)

Jumlah pelanggan: 50000
root
 |-- pelanggan_id: string (nullable = true)
 |-- nama: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- tanggal_daftar: string (nullable = true)
 |-- membership: string (nullable = true)

+------------+-----------+----------+--------------+----------+
|pelanggan_id|nama       |kota      |tanggal_daftar|membership|
+------------+-----------+----------+--------------+----------+
|1           |Pelanggan-1|Semarang  |2026-04-06    |Gold      |
|2           |Pelanggan-2|Temanggung|2025-06-22    |Silver    |
|3           |Pelanggan-3|Solo      |2026-05-28    |Silver    |
|4           |Pelanggan-4|Wonosobo  |2026-02-23    |Silver    |
|5           |Pelanggan-5|Magelang  |2026-01-19    |Silver    |
|6           |Pelanggan-6|Semarang  |2024-07-17    |Silver    |
|7           |Pelanggan-7|Semarang  |2024-03-16    |Silver    |
+------------+-----------+----------+--------------+----------+
only showing top 7 rows



In [54]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, ArrayType
skema_prod = StructType([
    StructField("produk_id", StringType()),
    StructField("nama_produk", StringType()),
    StructField("kategori", StringType()),
    StructField("harga_dasar", DoubleType()),
    StructField("garansi_bulan", LongType()),
    StructField("voltas", LongType()),
    StructField("ukuran", StringType()),
    StrkuctField("bahan", StringType()),
    StructField("kadaluarsa_hari", LongType()),
    StructField("penerbit", StringType()),
    StructField("tags", ArrayType(StringType())),
])

In [55]:
df_prod = spark.read.schema(skema_prod).json(f"{HDFS}/raw/produk/produk.json")
print("Jumlah produk:", df_prod.count())
df_prod.printSchema()
df_prod.show(7, truncate=False)

Jumlah produk: 5000
root
 |-- produk_id: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga_dasar: double (nullable = true)
 |-- garansi_bulan: long (nullable = true)
 |-- voltas: long (nullable = true)
 |-- ukuran: string (nullable = true)
 |-- bahan: string (nullable = true)
 |-- kadaluarsa_hari: long (nullable = true)
 |-- penerbit: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)

+---------+--------------+------------+-----------+-------------+------+------+---------+---------------+--------+------------------------+
|produk_id|nama_produk   |kategori    |harga_dasar|garansi_bulan|voltas|ukuran|bahan    |kadaluarsa_hari|penerbit|tags                    |
+---------+--------------+------------+-----------+-------------+------+------+---------+---------------+--------+------------------------+
|1        |Fashion-1     |Fashion     |350000.0   |NULL         |

**Pemisahan direktori raw dan curated**

Direktori raw digunkan untuk menyimpan dat asli sebelum proses pembersihan. SEdangkan direktori curted digunakan untuk menyimpan data yang telah melalui proses ETL dan pembersihan. Pemisahan ini bertujuan agar data  asli tetap terjaga dan proses pengolahan lebih terstuktur.

**Arti replikasi = 1**

Replikasi 1 berarti setiap blok hanya memiliki satu salinan. Jika terjadi kerusakan pada penyimpanan, tidak ada salinan lain yang bisa digunakan untuk menggantikan data tersebut. Sedangkan pada sistem dengan beberapa DataNode, data dapat disimpan dalam beberapa salinan agar lebih aman.

**Tahap 2 – Profiling Kualitas Data (Veracity)**

In [57]:
total_baris = df_trx.count()
print("total_baris :", total_baris)

total_baris : 2010000


In [60]:
order_unik = df_trx.select("order_id").distinct().count()
print("order_id unik", order_unik)

[Stage 52:>                                                         (0 + 2) / 2]

order_id unik 2000000


In [62]:
from pyspark.sql.functions import col

total = df_trx.count()

for c in df_trx.columns:
    kosong = df_trx.filter(col(c).isNull()).count()
    persen = (kosong / total) * 100
    print(c,":", kosong, "data kosong (", round(persen, 2), "%)")

order_id : 0 data kosong ( 0.0 %)


tanggal : 0 data kosong ( 0.0 %)


pelanggan_id : 0 data kosong ( 0.0 %)


produk_id : 0 data kosong ( 0.0 %)


qty : 6131 data kosong ( 0.31 %)


harga_satuan : 6003 data kosong ( 0.3 %)


... (2 outputs left out)

kota : 0 data kosong ( 0.0 %)


berat_kg : 0 data kosong ( 0.0 %)


jarak_km : 0 data kosong ( 0.0 %)


lama_kirim_hari : 0 data kosong ( 0.0 %)
status_pesanan : 0 data kosong ( 0.0 %)


In [66]:
harga_negatif = df_trx.filter(col("harga_satuan") < 0).count()
print("jumlah harga negatif:", harga_negatif)

[Stage 100:>                                                        (0 + 2) / 2]

jumlah harga negatif: 1968


In [77]:
print("Distribusi Qty")
df_trx.groupBy("qty").count().orderBy("qty").show()

Distribusi Qty


[Stage 118:>                                                        (0 + 2) / 2]

+-----+------+
|  qty| count|
+-----+------+
| NULL|  6131|
|  1.0|286837|
|  2.0|286145|
|  3.0|287568|
|  4.0|285605|
|  5.0|285267|
|  6.0|285425|
|  7.0|285999|
|999.0|  1023|
+-----+------+



In [72]:
df_qty = df_trx.select(col("qty").cast("double").alias("qty"))
kuartil_qty = df_qty.approxQuantile("qty", [0.25, 0.5, 0.75, 0.99, 0.999, 1.0], 0.001)

Q1 = kuartil_qty[0]
Q2 = kuartil_qty[1]
Q3 = kuartil_qty[2]

print("Q1:", Q1)
print("Q2 (median):", Q2)
print("Q3:", Q3)

[Stage 108:>                                                        (0 + 2) / 2]

Q1: 2.0
Q2 (median): 4.0
Q3: 6.0


In [73]:
IQR = Q3 - Q1
batas_bawah = Q1 - (1.5 * IQR)
batas_atas = Q3 + (1.5 * IQR)

print("IQR:", IQR)
print("Batas bawah:", batas_bawah)
print("Batas atas:", batas_atas)

IQR: 4.0
Batas bawah: -4.0
Batas atas: 12.0


In [75]:
from pyspark.sql.functions import col
jumlah_outlier_iqr = df_qty.filter(
    (col("qty") < batas_bawah) |
    (col("qty") > batas_atas)).count()

print("jumlah outlier IQR:", jumlah_outlier_iqr)

jumlah_qty_999 = df_trx.filter(col("qty") == 999).count()
print("jumlah_qty_999:", jumlah_qty_999)

jumlah outlier IQR: 1023
jumlah_qty_999: 1023


In [79]:
df_trx.groupBy("kota") \
  .count() \
  .orderBy("kota") \
  .show(100, truncate=False)

[Stage 121:>                                                        (0 + 2) / 2]

+----------+------+
|kota      |count |
+----------+------+
|KEBUMEN   |6280  |
|Kebumen   |238457|
|MAGELANG  |6379  |
|Magelang  |238449|
|PURWOREJO |6311  |
|Purworejo |238337|
|SEMARANG  |6254  |
|SOLO      |6335  |
|Semarang  |238446|
|Solo      |238603|
|TEMANGGUNG|6199  |
|Temanggung|238968|
|WONOSOBO  |6435  |
|Wonosobo  |239362|
|YOGYAKARTA|6340  |
|Yogyakarta|238364|
|kebumen   |6306  |
|magelang  |6240  |
|purworejo |6395  |
|semarang  |6374  |
|solo      |6367  |
|temanggung|6233  |
|wonosobo  |6277  |
|yogyakarta|6289  |
+----------+------+



In [82]:
hasil_kategori = df_prod.groupBy("kategori").count().orderBy("kategori")
hasil_kategori.show()

+------------+-----+
|    kategori|count|
+------------+-----+
|        Buku|  618|
|  Elektronik|  646|
|     Fashion|  615|
|   Kesehatan|  642|
|     Makanan|  604|
|    Olahraga|  631|
|    Otomotif|  594|
|Rumah Tangga|  650|
+------------+-----+



In [86]:
from pyspark.sql.functions import col, count, sum, when

hasil_atribut = df_prod.groupBy("kategori").agg(count("+").alias("total_produk").alias("total_produk"),
                (sum(when(col("garansi_bulan").isNotNull(),1).otherwise(0)) / count("*") * 100).alias("garansi_bulan"),
                (sum(when(col("voltas").isNotNull(),1).otherwise(0)) / count("*") * 100).alias("voltas"),
                (sum(when(col("ukuran").isNotNull(),1).otherwise(0)) / count("*") * 100).alias("ukuran"),
                (sum(when(col("bahan").isNotNull(),1).otherwise(0)) / count("*") * 100).alias("bahan"),
                (sum(when(col("kadaluarsa_hari").isNotNull(),1).otherwise(0)) / count("*") * 100).alias("kadaluarsa_hari"),
                (sum(when(col("penerbit").isNotNull(),1).otherwise(0)) / count("*") * 100).alias("penerbit"),

hasil_atribut.orderBy("kategori").show(truncate=False)

SyntaxError: incomplete input (3399577868.py, line 11)